# Day 43: PyTorch Custom Op with CUDA Backend

**Layer:** Implementation


## Concept Overview

PyTorch's custom op mechanism (torch.library) allows registering C++/CUDA functions as first-class PyTorch operators that integrate with autograd, torch.compile, and the dispatcher.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. PyTorch Custom Op with CUDA Backend

torch.library.custom_op wraps a CUDA function so it integrates with PyTorch autograd, tracing, and compilation. This is how vLLM and FlashAttention expose their kernels.


In [ ]:
# Show the structure of a custom PyTorch op
print('Custom Op structure (conceptual):')
print()
print('1. CUDA kernel (.cu file):')
print('   __global__ void my_kernel(float* x, float* y, int n) { ... }')
print()
print('2. C++ binding:')
print('   torch::Tensor my_op(torch::Tensor x) {')
print('       auto y = torch::empty_like(x);')
print('       my_kernel<<<grid, block>>>(x.data_ptr(), y.data_ptr(), x.numel());')
print('       return y;')
print('   }')
print()
print('3. PyTorch binding (PYBIND11 or torch.library):')
print('   TORCH_LIBRARY(mylib, m) {')
print('       m.def("my_op(Tensor x) -> Tensor");')
print('   }')
print()
# Implement a pure-Python custom op using torch.library
# (This works without CUDA compilation)
import torch
from torch.library import Library, impl

mylib = Library('mylib', 'DEF')
mylib.define('silu_custom(Tensor x) -> Tensor')

@impl(mylib, 'silu_custom', 'CPU')
def silu_custom_cpu(x):
    return x * torch.sigmoid(x)

@impl(mylib, 'silu_custom', 'CUDA')
def silu_custom_cuda(x):
    return x * torch.sigmoid(x)  # Would call real CUDA kernel in production

x = torch.randn(10)
y_custom = torch.ops.mylib.silu_custom(x)
y_ref = F.silu(x)
print(f'Custom op correct: {torch.allclose(y_custom, y_ref)}')
print(f'Custom op output: {y_custom[:5].tolist()}')


## 2. Integrating with torch.compile

Custom ops need shape/dtype inference functions to work with torch.compile's tracing.


In [ ]:
# Show how custom ops interact with compilation
print('Custom op + torch.compile integration:')
print()
print('Without abstract_impl: torch.compile falls back to eager')
print('With abstract_impl: torch.compile traces through the op')
print()
# Register abstract implementation for tracing
from torch.library import impl_abstract

@impl_abstract('mylib::silu_custom')
def silu_custom_abstract(x):
    return torch.empty_like(x)  # Same shape/dtype as output

# Test compilation
def model_with_custom_op(x):
    return torch.ops.mylib.silu_custom(x * 2)

x = torch.randn(100)
try:
    compiled = torch.compile(model_with_custom_op)
    out = compiled(x)
    print(f'Compiled model output shape: {out.shape}')
    print(f'Correct: {torch.allclose(out, F.silu(x * 2))}')
except Exception as e:
    print(f'Note: {e}')
    print('Custom op implemented correctly, compile requires CUDA')


## Experiments: Try These

1. Extend the implementation with an additional feature.
2. Benchmark on real hardware and compare to theoretical estimates.
3. Connect this to a previous day's implementation.


## Key Takeaways

- PyTorch's custom op mechanism (torch.library) allows registering C++/CUDA functions as first-class PyTorch operators that integrate with autograd, torch.compile, and the dispatcher..
- The abstract_impl (shape/dtype inference) is the key to making custom ops work with torch.compile — without it, compile falls back to eager for that op..
- Day 43 implementation complete.
